# Planejamento de rotas hospitalares

Este notebook executa o fluxo completo sem exigir um terminal. As etapas estão separadas para permitir a inspeção dos dados, a alteração dos parâmetros e a análise dos resultados. No VS Code, selecione o kernel da `.venv` e use **Executar tudo**.

## 1. Preparação do ambiente

A célula identifica a raiz do repositório e instala o projeto no próprio kernel. Ela pode ser executada novamente sem recriar o ambiente.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT), '--quiet'])
sys.path.insert(0, str(ROOT / 'src'))
print(f'Projeto preparado em: {ROOT}')

## 2. Importações

Os módulos são separados por responsabilidade: dados, otimização, comparação, visualização e relatórios.

In [ ]:
import json
import random
import statistics
import time
from IPython.display import IFrame, Markdown, clear_output, display
import matplotlib.pyplot as plt

from hospital_routes.baselines import brute_force, nearest_neighbor
from hospital_routes.genetic import GAConfig, GeneticOptimizer
from hospital_routes.io import load_problem, save_solution
from hospital_routes.models import Delivery, Depot, Problem, Vehicle
from hospital_routes.reporting import (GeminiReportGenerator, LocalReportGenerator, build_prompt, comparison_metrics, configured_gemini_key)
from hospital_routes.visualization import save_convergence_plot, save_route_map
print(f'Kernel: {sys.executable}')
print('Gemini configurado:', bool(configured_gemini_key()))

## 3. Configuração do experimento

Altere os valores abaixo e execute novamente a partir desta célula. `USAR_CENARIO_PADRAO = True` lê o arquivo do projeto. Com `False`, um novo cenário reproduzível é criado a partir da semente. O elitismo aceita zero.

In [ ]:
USAR_CENARIO_PADRAO = True
NUMERO_ENTREGAS = 10
NUMERO_VEICULOS = 3
CAPACIDADE_KG = 55.0
AUTONOMIA_KM = 65.0

TAMANHO_POPULACAO = 120
GERACOES = 300
TAXA_CROSSOVER = 0.90
TAXA_MUTACAO = 0.20
ELITISMO = 4
SEMENTE = 42

## 4. Construção do cenário

Cada ponto representa uma entrega. O hospital é o depósito de saída e retorno e não entra nessa contagem. Coordenadas, demandas e prioridades podem ser alteradas diretamente no objeto `problem` após esta célula.

In [ ]:
if USAR_CENARIO_PADRAO:
    problem = load_problem(ROOT / 'data' / 'deliveries.json')
else:
    rng = random.Random(SEMENTE)
    depot = Depot('Hospital Central', rng.uniform(-13.005, -12.94), rng.uniform(-38.53, -38.45))
    deliveries = tuple(
        Delivery(
            id=f'E{i + 1:02d}', name=f'Ponto {i + 1:02d}',
            latitude=rng.uniform(-13.05, -12.89), longitude=rng.uniform(-38.57, -38.33),
            demand_kg=round(rng.uniform(4, CAPACIDADE_KG * 0.30), 1),
            priority=rng.randint(1, 3), service_minutes=10,
        ) for i in range(NUMERO_ENTREGAS)
    )
    vehicles = tuple(Vehicle(f'VEIC-{i + 1:02d}', CAPACIDADE_KG, AUTONOMIA_KM) for i in range(NUMERO_VEICULOS))
    problem = Problem(depot, deliveries, vehicles)

print(f'Hospital: {problem.depot.latitude:.6f}, {problem.depot.longitude:.6f}')
print(f'Entregas: {len(problem.deliveries)} | Veículos: {len(problem.vehicles)}')
for delivery in problem.deliveries:
    print(f'{delivery.id}: ({delivery.latitude:.6f}, {delivery.longitude:.6f}) | {delivery.demand_kg:.1f} kg | prioridade {delivery.priority}')

### 4.1 Diagnóstico dos dados e viabilidade mínima

Antes da busca, são verificadas condições necessárias de capacidade. Elas não substituem a otimização, pois a distribuição entre veículos e a autonomia permanecem dependentes das rotas.

In [ ]:
total_demand = sum(item.demand_kg for item in problem.deliveries)
total_capacity = sum(vehicle.capacity_kg for vehicle in problem.vehicles)
largest_demand = max(item.demand_kg for item in problem.deliveries)
largest_capacity = max(vehicle.capacity_kg for vehicle in problem.vehicles)
diagnostics = {
    'demanda_total_kg': total_demand,
    'capacidade_total_kg': total_capacity,
    'capacidade_agregada_suficiente': total_demand <= total_capacity,
    'maior_entrega_atendivel': largest_demand <= largest_capacity,
}
diagnostics

## 5. Configuração do algoritmo genético

A representação é uma permutação das entregas. A função de aptidão combina distância, atendimento prioritário e penalidades por excesso de carga ou autonomia.

In [ ]:
config = GAConfig(
    population_size=TAMANHO_POPULACAO, generations=GERACOES,
    crossover_rate=TAXA_CROSSOVER, mutation_rate=TAXA_MUTACAO,
    elite_size=ELITISMO, seed=SEMENTE,
)
optimizer = GeneticOptimizer(problem, config)
baseline = nearest_neighbor(optimizer)
print(f'Fitness do vizinho mais próximo: {baseline.fitness:.2f}')

## 6. Evolução e acompanhamento da convergência

Os gráficos possuem escalas independentes. O primeiro acompanha o melhor indivíduo; o segundo acompanha a média da população. A saída é atualizada periodicamente durante a evolução.

In [ ]:
solution = None
for stats, solution in optimizer.evolve():
    if stats.generation % 10 == 0 or stats.generation == GERACOES - 1:
        clear_output(wait=True)
        generations = [item.generation for item in optimizer.history]
        fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
        axes[0].plot(generations, [item.best for item in optimizer.history], color='#24966f')
        axes[0].set_title('Melhor fitness'); axes[0].set_ylabel('Fitness'); axes[0].grid(alpha=.25)
        axes[1].plot(generations, [item.mean for item in optimizer.history], color='#657f9f')
        axes[1].set_title('Fitness médio'); axes[1].set_xlabel('Geração'); axes[1].set_ylabel('Fitness'); axes[1].grid(alpha=.25)
        fig.tight_layout(); display(fig); plt.close(fig)
        print(f'Geração {stats.generation} | melhor: {stats.best:.2f} | média: {stats.mean:.2f}')

assert solution is not None
print(f'Concluído em {len(optimizer.history)} gerações.')

## 7. Análise da solução

A comparação usa a mesma função de aptidão. A viabilidade exige que todas as rotas respeitem capacidade e autonomia.

In [ ]:
improvement = 100 * (baseline.fitness - solution.fitness) / baseline.fitness
print(f'Fitness final: {solution.fitness:.2f}')
print(f'Distância total: {solution.total_distance_km:.2f} km')
print(f'Solução viável: {solution.feasible}')
print(f'Redução frente ao vizinho mais próximo: {improvement:.2f}%')
for i, route in enumerate(solution.routes):
    metric = solution.metrics[i]
    stops = ' → '.join(problem.deliveries[j].id for j in route) or 'não utilizado'
    print(f'{problem.vehicles[i].id}: {stops} | {metric.distance_km:.2f} km | {metric.load_kg:.1f} kg')
comparison = comparison_metrics(solution, baseline)
comparison

## 8. Comparação experimental e robustez

A força bruta é executada em uma instância reduzida de oito entregas para manter o custo fatorial controlado. Em seguida, cinco sementes avaliam a variabilidade do algoritmo genético. Os tempos dependem do equipamento e devem ser interpretados apenas dentro desta execução.

In [ ]:
small_problem = Problem(problem.depot, problem.deliveries[:8], problem.vehicles)
small_optimizer = GeneticOptimizer(
    small_problem, GAConfig(population_size=80, generations=150, stagnation_limit=50, seed=42)
)
times = {}
start = time.perf_counter(); nearest_small = nearest_neighbor(small_optimizer); times['vizinho'] = time.perf_counter() - start
start = time.perf_counter(); exact_small = brute_force(small_optimizer, max_deliveries=8); times['forca_bruta'] = time.perf_counter() - start
start = time.perf_counter(); genetic_small = small_optimizer.run(); times['genetico'] = time.perf_counter() - start
print('Instância reduzida (8 entregas)')
for name, item in [('Vizinho mais próximo', nearest_small), ('Força bruta', exact_small), ('Algoritmo genético', genetic_small)]:
    key = {'Vizinho mais próximo':'vizinho','Força bruta':'forca_bruta','Algoritmo genético':'genetico'}[name]
    print(f'{name}: fitness={item.fitness:.2f}; distância={item.total_distance_km:.2f} km; tempo={times[key]:.4f} s')
print('GA atingiu o fitness da força bruta:', abs(genetic_small.fitness - exact_small.fitness) < 1e-9)

In [ ]:
seed_results = []
for seed in (7, 21, 42, 84, 123):
    experiment = GeneticOptimizer(
        problem, GAConfig(population_size=80, generations=150, stagnation_limit=50, seed=seed)
    )
    reference = nearest_neighbor(experiment)
    start = time.perf_counter(); result = experiment.run(); elapsed = time.perf_counter() - start
    reduction = 100 * (reference.fitness - result.fitness) / reference.fitness
    seed_results.append((seed, result.fitness, result.total_distance_km, result.feasible, elapsed, reduction))
for row in seed_results:
    print(f'seed={row[0]} | fitness={row[1]:.2f} | distância={row[2]:.2f} km | viável={row[3]} | tempo={row[4]:.3f} s | redução={row[5]:.2f}%')
print(f'Fitness médio: {statistics.mean(r[1] for r in seed_results):.2f}')
print(f'Desvio-padrão do fitness: {statistics.pstdev(r[1] for r in seed_results):.2f}')
print(f'Redução média frente à referência: {statistics.mean(r[5] for r in seed_results):.2f}%')

## 9. Geração dos entregáveis

Esta etapa salva o JSON, o mapa HTML, o gráfico final e o relatório operacional na pasta `outputs`.

In [ ]:
OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(exist_ok=True)
save_solution(OUTPUT / 'solution.json', problem, solution)
save_route_map(problem, solution, OUTPUT / 'routes_map.html')
save_convergence_plot(optimizer.history, OUTPUT / 'convergence.png')
try:
    report = GeminiReportGenerator().generate(problem, solution, comparison=comparison)
    report_source = 'Gemini'
except RuntimeError as error:
    report = LocalReportGenerator().generate(problem, solution, comparison=comparison)
    report_source = f'gerador local; Gemini indisponível: {error}'
(OUTPUT / 'daily_report.md').write_text(report, encoding='utf-8')
(OUTPUT / 'comparison.json').write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Origem do relatório: {report_source}')
print('Arquivos gerados:')
for name in ('solution.json', 'routes_map.html', 'convergence.png', 'daily_report.md', 'comparison.json'):
    print(' -', OUTPUT / name)

## 10. Mapa e relatório no notebook

O mapa interativo e as instruções operacionais são apresentados abaixo sem abrir outro programa.

In [ ]:
display(IFrame(src=str(OUTPUT / 'routes_map.html'), width='100%', height=550))
display(Markdown(report))

## 11. Integração com a LLM

A LLM recebe somente a solução estruturada e o comparativo mensurável. Ela não otimiza rotas. O prompt proíbe estimativas de tempo ou dinheiro sem dados observados e a saída deve ser revisada por uma pessoa responsável.

In [ ]:
prompt_preview = build_prompt(problem, solution, comparison=comparison)
print('Caracteres do prompt:', len(prompt_preview))
print('Instruções de controle presentes:', all(term in prompt_preview for term in ('Não invente', 'Não converta', 'Contexto estruturado')))
print('O contexto enviado contém apenas dados operacionais fictícios e a solução calculada.')

### 11.1 Relatório semanal e perguntas em linguagem natural

As chamadas abaixo são opcionais para controlar uso de cota. Quando habilitadas, usam o mesmo contexto fundamentado da rota. O fallback local preserva a execução do notebook, mas não é apresentado como uma LLM.

In [ ]:
EXECUTAR_TAREFAS_ADICIONAIS_LLM = False
if EXECUTAR_TAREFAS_ADICIONAIS_LLM:
    generator = GeminiReportGenerator()
    weekly_report = generator.generate(problem, solution, task='weekly', comparison=comparison)
    question = 'Quais veículos atendem entregas de prioridade crítica?'
    answer = generator.generate(problem, solution, task=f'Responda: {question}', comparison=comparison)
    display(Markdown(weekly_report))
    display(Markdown('**Pergunta:** ' + question + '\n\n**Resposta:** ' + answer))
else:
    print('Chamadas adicionais desativadas. Altere a variável para demonstrar relatório semanal e perguntas.')

## 12. Síntese metodológica

A análise deve separar três níveis: (1) validade computacional, verificada por testes; (2) qualidade experimental, avaliada por referências e múltiplas sementes; e (3) validade operacional, limitada pela qualidade das distâncias, demandas e restrições fornecidas. A solução é adequada para demonstração e experimentação, mas requer matriz viária, tempos históricos e governança de dados antes de uso em operações reais.